<a href="https://colab.research.google.com/github/Natalie-Bojorquez/CIS3920Group4/blob/main/CIS3920_Group4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [3920] Career Change Prediction
Data file:
* https://raw.githubusercontent.com/vjavaly/Baruch-CIS-3920/refs/heads/main/data/career_change_prediction_100K.csv

In [ ]:
from datetime import datetime
print(f'Run time: {datetime.now().strftime("%D %T")}')

In [ ]:
# Import libraries

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

from imblearn.over_sampling import SMOTE  # SMOTE

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Cross Validation
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold

# Evaluation Metrics
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance


from sklearn.linear_model import LogisticRegression # Logistic Regression
from sklearn.tree import export_graphviz
from sklearn.tree import DecisionTreeClassifier  # base model for AdaBoost Classifier
from sklearn.ensemble import RandomForestClassifier # Random Forest

from sklearn.ensemble import AdaBoostClassifier  # Adaptive Boosting
from sklearn.ensemble import GradientBoostingClassifier # Gradient Boosting
from sklearn.svm import SVC #Support Vector Classifier


#### Career Change Prediction
Business Problem  
... [enter business problem description]

Columns [...in context of business problem]
* Employee ID: Unique identifier for the employee
* Employer ID:
* Field of Study
* Current Occupation
* Age
* Gender
* Years of Experience
* Education Level
* Industry Growth Rate
* Job Satisfaction Level
* Work-Life Balance
* Job Opportunities
* Salary
* Job Security Level
* Interested in Career Change
* Skills Gap
* Family Influence Level
* Has Mentor
* Has Certification
* Has Freelancing Experience
* Willing to Relocate
* Professional Network Strength
* '# Career Change Events'
* Technical Proficiency Level
* Likely to Change Occupation  

# Exploratory Data Analysis

### Load data

In [ ]:
# Read data file into a Pandas DataFrame
# CSV file format
# Field separator is ','
raw_df = pd.read_csv('https://raw.githubusercontent.com/vjavaly/Baruch-CIS-3920/refs/heads/main/data/career_change_prediction_100K.csv',
                 sep=',')

### Examine data

In [ ]:
# Display DataFrame dimensions
raw_df.shape  # 100,000 data instances (n rows), 25 features (m columns)

In [ ]:
# Display DataFrame info: entries, columns, non-null counts, data types
raw_df.info()

In [ ]:
# Preview first 5 rows
raw_df.head()

In [ ]:
# Preview last 5 rows
raw_df.tail()

In [ ]:
# Display random sample of N rows
raw_df.sample(5)

In [ ]:
# Separate numeric and categorical variables
numeric_raw_df = raw_df.select_dtypes(include=[np.number])
object_raw_df = raw_df.select_dtypes(exclude=[np.number])

In [ ]:
# Inspect numeric variable unique values
for col in numeric_raw_df.columns:
  print(f'{col}: {numeric_raw_df[col].sort_values().unique()}', end = '\n''\n')

In [ ]:
# Inspect categorical variables unique values
for col in object_raw_df.columns:
  print(f'{col}: {object_raw_df[col].unique()}', end = '\n''\n')


In [ ]:
# Display summary statistics of numeric columns ; *.describe() method excludes null values*
raw_df.describe()

# Data Preparation

In [ ]:
# Create a second DataFrame for data preparation
df = raw_df.copy()

### Identify & Handle Duplicates

In [ ]:
# Check for duplicate rows across all columns
total_duplicates = df.duplicated().sum()
print(f'Dataset contains {total_duplicates} duplicate rows')

In [ ]:
# Remove duplicate rows
df.drop_duplicates(inplace=True)

In [ ]:
# Confirm no duplicate rows remain
total_duplicates = df.duplicated().sum()
print(f'Dataset contains {total_duplicates} duplicate rows')

In [ ]:
# Display updated DataFrame dimensions
df.shape  # 99,747 data instances

###Handle Identifiers
- Employee ID
- Employer ID

In [ ]:
# Check if Employee ID is unique for every observation
df.shape[0] == df['Employee ID'].nunique()

In [ ]:
# Set Employee ID as the index to uniquely identify each observation
df.set_index('Employee ID', inplace=True)

# Randomly inspect 5 observations
df.sample(5)

In [ ]:
# Evaluate whether Employer ID is useful for predicting occupation change
pd.crosstab(
    df['Employer ID'],
    df['Likely to Change Occupation'],
    normalize='index'
    )*100             # Ocupation-change rates are relatively consistent across employers, suggesting limited predictive value

Ocupation-change rates are relatively consistent across employers, suggesting limited predictive value

In [ ]:
# Drop Employer ID from DataFrame
df.drop(columns='Employer ID', inplace=True)

# Randomly inspect 5 observations
df.sample(5)

### Identify & Handle Null Values

In [ ]:
# Count missing values in each column
df.isnull().sum()

In [ ]:
# Drop Family Influence because too many null values (25,293 out of 99,747)
pd.set_option('display.max_columns', None)
df = df.drop('Family Influence Level', axis =1)

# Randomly Inspect 5 observations
df.sample(5)



---



# Split training and test sets to avoid data leakage

In [ ]:
# Independent variables: All columns except 'Likely to Change Occupation'
X = df.drop('Likely to Change Occupation', axis = 1)

# Dependent variable: 'Likely to Change Occupation'
y = df['Likely to Change Occupation']

In [ ]:
# Split data into training and 25% test sets
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                                stratify=y,    # preserves class distribution in both sets
                                                                test_size=0.25,
                                                                random_state=42)  # makes randomized split reproducible

In [ ]:
# Merge X_train and y_train into a single dataframe
train_set = X_train.join(y_train)

train_set.sample(5)

In [ ]:
# Merge X_test and y_test into a single dataframe
test_set = X_test.join(y_test)

test_set.sample(5)



---



Handle numeric column nulls

In [ ]:
# Check for missing values in numeric columns
df.select_dtypes(include=[np.number]).isnull().sum()

Numeric columns with nulls:
  - 'Age' (34)
  - 'Salary' (345)
  - '# Job Opportunities' (390)
   - 'Years of Experience' (484)

In [ ]:
# Generate summary statistics for numeric null columns in transposed format *null values are excluded in .describe()*
df[['Age','Years of Experience','Salary','# Job Opportunities']].describe().T    # .T -> rotates the data 90 degrees

In [ ]:
# Display skewed distribution of Salary by plotting a histogram with a Kernel Distribution Estimation Plot overlay
plt.figure(figsize=[6,4])
sns.set(style="ticks")
sns.histplot(data=df, x='Salary', kde=True)  # kde=True -> draw curve fit to shape... bell curve in this case
plt.title('Salary Distribution')
plt.show()

Distribution of Salary is skewed to the right. Impute median for Salary nulls.

In [ ]:
# Replace missing numeric values in train set with an arithmetic (mean or median) strategy for each class
# mean strategy
train_set['Age'] = train_set['Age'].fillna(train_set.groupby('Likely to Change Occupation')['Age'].transform('mean'))
train_set['Years of Experience'] = train_set['Years of Experience'].fillna(train_set.groupby('Likely to Change Occupation')['Years of Experience'].transform('mean'))
train_set['# Job Opportunities'] = train_set['# Job Opportunities'].fillna(train_set.groupby('Likely to Change Occupation')['# Job Opportunities'].transform('mean'))

# median strategy
train_set['Salary'] = train_set['Salary'].fillna(train_set.groupby('Likely to Change Occupation')['Salary'].transform('median'))

In [ ]:
# Replace missing numeric values in test set with an arithmetic (mean or median) strategy for each class
# mean strategy
test_set['Age'] = test_set['Age'].fillna(test_set.groupby('Likely to Change Occupation')['Age'].transform('mean'))
test_set['Years of Experience'] = test_set['Years of Experience'].fillna(test_set.groupby('Likely to Change Occupation')['Years of Experience'].transform('mean'))
test_set['# Job Opportunities'] = test_set['# Job Opportunities'].fillna(test_set.groupby('Likely to Change Occupation')['# Job Opportunities'].transform('mean'))

# median strategy
test_set['Salary'] = test_set['Salary'].fillna(test_set.groupby('Likely to Change Occupation')['Salary'].transform('median'))

Handle categorical column nulls

In [ ]:
# Check for missing values in categorical columns
df.select_dtypes(include=[np.number]).isnull().sum()

Categorical columns with nulls:
  - 'Gender' (205)
  - 'Education Level' (303)
  - 'Industry Growth Rate' (598)
  - 'Job Satisfaction Level' (471)
  - 'Work-Life Balance' (397)
  - 'Has Mentor' (730)
  - 'Has Certification' (633)

In [ ]:
# Replace missing non-numeric values with a most_frequent (mode) strategy for each class
train_set['Gender'] = train_set['Gender'].fillna(train_set.groupby('Likely to Change Occupation')['Gender'].transform(lambda x: x.mode()[0]))
train_set['Education Level'] = train_set['Education Level'].fillna(train_set.groupby('Likely to Change Occupation')['Education Level'].transform(lambda x: x.mode()[0]))
train_set['Industry Growth Rate'] = train_set['Industry Growth Rate'].fillna(train_set.groupby('Likely to Change Occupation')['Industry Growth Rate'].transform(lambda x: x.mode()[0]))
train_set['Job Satisfaction Level'] = train_set['Job Satisfaction Level'].fillna(train_set.groupby('Likely to Change Occupation')['Job Satisfaction Level'].transform(lambda x: x.mode()[0]))
train_set['Work-Life Balance'] = train_set['Work-Life Balance'].fillna(train_set.groupby('Likely to Change Occupation')['Work-Life Balance'].transform(lambda x: x.mode()[0]))
train_set['Has Mentor'] = train_set['Has Mentor'].fillna(train_set.groupby('Likely to Change Occupation')['Has Mentor'].transform(lambda x: x.mode()[0]))
train_set['Has Certification'] = train_set['Has Certification'].fillna(train_set.groupby('Likely to Change Occupation')['Has Certification'].transform(lambda x: x.mode()[0]))

In [ ]:
# Replace missing non-numeric values with a most_frequent (mode) strategy for each class
test_set['Gender'] = test_set['Gender'].fillna(test_set.groupby('Likely to Change Occupation')['Gender'].transform(lambda x: x.mode()[0]))
test_set['Education Level'] = test_set['Education Level'].fillna(test_set.groupby('Likely to Change Occupation')['Education Level'].transform(lambda x: x.mode()[0]))
test_set['Industry Growth Rate'] = test_set['Industry Growth Rate'].fillna(test_set.groupby('Likely to Change Occupation')['Industry Growth Rate'].transform(lambda x: x.mode()[0]))
test_set['Job Satisfaction Level'] = test_set['Job Satisfaction Level'].fillna(test_set.groupby('Likely to Change Occupation')['Job Satisfaction Level'].transform(lambda x: x.mode()[0]))
test_set['Work-Life Balance'] = test_set['Work-Life Balance'].fillna(test_set.groupby('Likely to Change Occupation')['Work-Life Balance'].transform(lambda x: x.mode()[0]))
test_set['Has Mentor'] = test_set['Has Mentor'].fillna(test_set.groupby('Likely to Change Occupation')['Has Mentor'].transform(lambda x: x.mode()[0]))
test_set['Has Certification'] = test_set['Has Certification'].fillna(test_set.groupby('Likely to Change Occupation')['Has Certification'].transform(lambda x: x.mode()[0]))

In [ ]:
# Check again if there are any missing values
train_set.isnull().any()

In [ ]:
# Check again if there are any missing values
test_set.isnull().any()

###Identify & Handle Outliers

In [ ]:
# Boxplot for Age across classes
plt.figure(figsize=(5, 4))
sns.boxplot(x='Likely to Change Occupation',
            y='Age',
            data=df,
            palette='muted',
            hue='Likely to Change Occupation')

plt.title('Age Distribution Across Classes')
plt.xlabel('Likely to Change Occupation', fontsize=10)
plt.ylabel('Age', fontsize=10)

# Display the plot
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot for Age across classes
plt.figure(figsize=(5, 4))
sns.boxplot(x='Likely to Change Occupation',
            y='Years of Experience',
            data=df,
            palette='muted',
            hue='Likely to Change Occupation')

plt.title('Years of Experience Distribution Across Classes')
plt.xlabel('Likely to Change Occupation', fontsize=10)
plt.ylabel('Years of Experience', fontsize=10)

# Display the plot
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot for Salary across classes
plt.figure(figsize=(5, 4))
sns.boxplot(x='Likely to Change Occupation',
            y='Salary',
            data=df,
            palette='muted',
            hue='Likely to Change Occupation')

plt.title('Salary Distribution Across Classes')
plt.xlabel('Likely to Change Occupation', fontsize=10)
plt.ylabel('Salary', fontsize=10)

# Display the plot
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot for Salary across classes
plt.figure(figsize=(5, 4))
sns.boxplot(x='Likely to Change Occupation',
            y='# Job Opportunities',
            data=df,
            palette='muted',
            hue='Likely to Change Occupation')

plt.title('# Job Opportunities Distribution Across Classes')
plt.xlabel('Likely to Change Occupation', fontsize=10)
plt.ylabel('# Job Opportunities', fontsize=10)

# Display the plot
plt.tight_layout()
plt.show()

No outliers detected.

####Check correlation amongst columns

In [ ]:
# Correlation Matrix

numeric_df = df.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()

plt.figure(figsize=(10, 7))

sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    vmin=-1,
    vmax=1,
    center=0
)

plt.title('Correlation Matrix of Numeric Variables')
plt.tight_layout()
plt.show()

No multicollinearity detected amongst numeric columns

####Visualize Class Distribution

In [ ]:
# Bar Chart of class distribution (target variable: Likely to Change Occupation)
cd = raw_df['Likely to Change Occupation'].value_counts().plot.bar(title='Class Distribution', ylabel='Count')
cd.bar_label(cd.containers[0])
plt.show()


In [ ]:
# Generate class distribution percentages
likely = 91784
unlikely = 8216

print(f'Majority Class: {likely/99747 *100: .3f}%')
print(f'Minority Class: {unlikely/99747 *100: .3f}%')

Large class imbalance: Likely is the Majority (~ 92.02%) & Unlikely is the Minority (~ 8.24%)

### Variable Encoding

In [ ]:
# Filter categorical variables
object_df = df.select_dtypes(exclude=[np.number])

# Display categorical variable unique values
for col in object_df.columns:
  print(f'{col}: {object_df[col].unique()}', end = '\n''\n')

####Ordinal Encoding using OrdinalEncoder
- Must specify categories in logical order

In [ ]:
# Arrange values according to relative value within a dictionary
CATEGORIES = {
    'Education Level': [['High School', "Bachelor's", "Master's",'PhD']],
    'Industry Growth Rate': [['Low', 'Medium', 'High']],
    'Job Satisfaction Level': [['Low', 'Moderate', 'High']],
    'Work-Life Balance': [['Poor', 'Good', 'Moderate', 'Excellent']],
    'Job Security Level': [['Poor', 'Moderate', 'Excellent']],
    'Skills Gap': [['Low', 'Moderate', 'High']],
    'Professional Network Strength': [['Weak', 'Average', 'Strong']],
    'Technical Proficiency Level': [['Beginner', 'Intermediate', 'Expert']]
    }

In [ ]:
# Initialize the OrdinalEncoder for Education Level
oe = OrdinalEncoder(categories=CATEGORIES['Education Level'])

# Generate encodings
train_set['Education Level'] = oe.fit_transform(train_set[['Education Level']])
test_set['Education Level'] = oe.fit_transform(test_set[['Education Level']])

In [ ]:
# Initialize the OrdinalEncoder for Industry Growth Rate
oe = OrdinalEncoder(categories=CATEGORIES['Industry Growth Rate'])

# Generate encodings
train_set['Industry Growth Rate'] = oe.fit_transform(train_set[['Industry Growth Rate']])
test_set['Industry Growth Rate'] = oe.fit_transform(test_set[['Industry Growth Rate']])

In [ ]:
# Initialize the OrdinalEncoder for Job Satisfaction Level
oe = OrdinalEncoder(categories=CATEGORIES['Job Satisfaction Level'])

# Generate encodings
train_set['Job Satisfaction Level'] = oe.fit_transform(train_set[['Job Satisfaction Level']])
test_set['Job Satisfaction Level'] = oe.fit_transform(test_set[['Job Satisfaction Level']])

In [ ]:
# Initialize the OrdinalEncoder for Work-Life Balance
oe = OrdinalEncoder(categories=CATEGORIES['Work-Life Balance'])

# Generate encodings
train_set['Work-Life Balance'] = oe.fit_transform(train_set[['Work-Life Balance']])
test_set['Work-Life Balance'] = oe.fit_transform(test_set[['Work-Life Balance']])

In [ ]:
# Initialize the OrdinalEncoder for Job Security Level
oe = OrdinalEncoder(categories=CATEGORIES['Job Security Level'])

# Generate encodings
train_set['Job Security Level'] = oe.fit_transform(train_set[['Job Security Level']])
test_set['Job Security Level'] = oe.fit_transform(test_set[['Job Security Level']])

In [ ]:
# Initialize the OrdinalEncoder for Skills Gap
oe = OrdinalEncoder(categories=CATEGORIES['Skills Gap'])

# Generate encodings
train_set['Skills Gap'] = oe.fit_transform(train_set[['Skills Gap']])
test_set['Skills Gap'] = oe.fit_transform(test_set[['Skills Gap']])

In [ ]:
# Initialize the OrdinalEncoder for Professional Network Strength
oe = OrdinalEncoder(categories=CATEGORIES['Professional Network Strength'])

# Generate encodings
train_set['Professional Network Strength'] = oe.fit_transform(train_set[['Professional Network Strength']])
test_set['Professional Network Strength'] = oe.fit_transform(test_set[['Professional Network Strength']])

In [ ]:
# Initialize the OrdinalEncoder for Technical Proficiency Level
oe = OrdinalEncoder(categories=CATEGORIES['Technical Proficiency Level'])

# Generate encodings
train_set['Technical Proficiency Level'] = oe.fit_transform(train_set[['Technical Proficiency Level']])
test_set['Technical Proficiency Level'] = oe.fit_transform(test_set[['Technical Proficiency Level']])

In [ ]:
# Display a sample of 10 rows from updated train_set
train_set.sample(10)

In [ ]:
# Display a sample of 10 rows from updated test_set
test_set.sample(10)

####Nominal Encoding using OneHotEncoder or get_dummies

Field of Study:
- ['Biology' 'Economics' 'Medicine' 'Computer Science' 'Psychology'
 'Business' 'Education' 'Law' 'Mechanical Engineering' 'Arts']

Current Occupation:
- ['Economist' 'Doctor' 'Psychologist' 'Lawyer' 'Biologist' 'Teacher'
 'Artist' 'Business Analyst' 'Software Developer' 'Mechanical Engineer']

Gender: ['Female' 'Male']

Interested in Career Change: ['No' 'Yes']

Has Mentor: ['N' 'Y']

Has Certification: ['N' 'Y']

Has Freelancing Experience: ['N' 'Y']

Willing to Relocate: ['No' 'Yes']

In [ ]:
# Encode Field of Study using get_dummies
train_set = pd.get_dummies(train_set, columns=['Field of Study'],
                    drop_first=True,          # to reduce multicollinearity in linear models
                    dtype=int)                # int to represent binary indicators (1-present, 0-absent)

test_set = pd.get_dummies(test_set, columns=['Field of Study'],
                    drop_first=True,
                    dtype=int)

In [ ]:
# Encode Current Occupation using get_dummies
train_set = pd.get_dummies(train_set, columns=['Current Occupation'],
                    drop_first=True,          # to reduce multicollinearity in linear models
                    dtype=int)                # int to represent binary indicators (1-present, 0-absent)

test_set = pd.get_dummies(test_set, columns=['Current Occupation'],
                    drop_first=True,
                    dtype=int)

In [ ]:
# Encode Gender using get_dummies
train_set = pd.get_dummies(train_set, columns=['Gender'],
                    drop_first=True,          # to reduce multicollinearity in linear models
                    dtype=int)                # int to represent binary indicators (1-present, 0-absent)

test_set = pd.get_dummies(test_set, columns=['Gender'],
                    drop_first=True,
                    dtype=int)

In [ ]:
# Encode Interested in Career Change using get_dummies
train_set = pd.get_dummies(train_set, columns=['Interested in Career Change'],
                    drop_first=True,          # to reduce multicollinearity in linear models
                    dtype=int)                # int to represent binary indicators (1-present, 0-absent)

test_set = pd.get_dummies(test_set, columns=['Interested in Career Change'],
                    drop_first=True,
                    dtype=int)

In [ ]:
# Encode Has Mentor using get_dummies
train_set = pd.get_dummies(train_set, columns=['Has Mentor'],
                    drop_first=True,          # to reduce multicollinearity in linear models
                    dtype=int)                # int to represent binary indicators (1-present, 0-absent)

test_set = pd.get_dummies(test_set, columns=['Has Mentor'],
                    drop_first=True,
                    dtype=int)

In [ ]:
# Encode Has Certification using get_dummies
train_set = pd.get_dummies(train_set, columns=['Has Certification'],
                    drop_first=True,          # to reduce multicollinearity in linear models
                    dtype=int)                # int to represent binary indicators (1-present, 0-absent)

test_set = pd.get_dummies(test_set, columns=['Has Certification'],
                    drop_first=True,
                    dtype=int)

In [ ]:
# Encode Has Freelancing Experience using get_dummies
train_set = pd.get_dummies(train_set, columns=['Has Freelancing Experience'],
                    drop_first=True,          # to reduce multicollinearity in linear models
                    dtype=int)                # int to represent binary indicators (1-present, 0-absent)

test_set = pd.get_dummies(test_set, columns=['Has Freelancing Experience'],
                    drop_first=True,
                    dtype=int)

In [ ]:
# Encode Willing to Relocate using get_dummies
train_set = pd.get_dummies(train_set, columns=['Willing to Relocate'],
                    drop_first=True,          # to reduce multicollinearity in linear models
                    dtype=int)                # int to represent binary indicators (1-present, 0-absent)

test_set = pd.get_dummies(test_set, columns=['Willing to Relocate'],
                    drop_first=True,
                    dtype=int)

In [ ]:
# Display a sample of 5 rows from updated train_set
train_set.sample(5)

In [ ]:
# Display a sample of 5 rows from updated test_set
test_set.sample(5)

In [ ]:
# Display updated DataFrame Dimensions
train_set.shape

In [ ]:
# Display updated DataFrame Dimensions
test_set.shape

####Label Encoding using LabelEncoder

In [ ]:
# Label encode Likely to Change Occupation
le = LabelEncoder()
train_set['Likely to Change Occupation'] = le.fit_transform(train_set['Likely to Change Occupation'])
test_set['Likely to Change Occupation'] = le.fit_transform(test_set['Likely to Change Occupation'])

In [ ]:
# Identify the positive class
le.classes_[1] # => 'Unlikely'

In [ ]:
# Display a sample of 5 rows from updated train_set
train_set.sample(5)

In [ ]:
# Display a sample of 5 rows from updated test_set
test_set.sample(5)

With variable encoding complete, all features are numerical

####Data Discretization (assess if necessary after first model iteration)

###Feature Scaling using StandardScaler
- Standardize only the continuous, non-binary independent variables

In [ ]:
# Separate independent and dependent variables from updated train_set

# Independent variables: All columns except 'Likely to Change Occupation'
X_train = train_set.drop('Likely to Change Occupation', axis = 1)

# Dependent variable: 'Likely to Change Occupation'
y_train = train_set['Likely to Change Occupation']

In [ ]:
# Separate independent and dependent variables from updated test_set

# Independent variables: All columns except 'Likely to Change Occupation'
X_test = test_set.drop('Likely to Change Occupation', axis = 1)

# Dependent variable: 'Likely to Change Occupation'
y_test = test_set['Likely to Change Occupation']

In [ ]:
# Define the columns to be scaled (EXCLUDE binary features)
scale_cols = ['Education Level','Industry Growth Rate',
              'Job Satisfaction Level','Work-Life Balance',
              'Job Security Level','Skills Gap',
              'Professional Network Strength','Technical Proficiency Level',
              'Years of Experience','# Job Opportunities',
              'Salary','# Career Change Events',
              'Age']

# Standardize columns and overwrite them in X
X_train[scale_cols] = StandardScaler().fit_transform(X_train[scale_cols])
X_test[scale_cols] = StandardScaler().fit_transform(X_test[scale_cols])

In [ ]:
# Display 5 sample rows from final set of independent variables
X_train.sample(5)

In [ ]:
# Display 5 sample rows from final set of independent variables
X_test.sample(5)

###PCA (assess if necessary after first model iteration)
- standardization prefered for feature scaling

# First Iteration

##Train Logistic Regression model 1

In [ ]:
# Instantiate logistic regression
lr_1 = LogisticRegression(random_state=42)

# Check model's default hyperparameters
lr_1.get_params()

In [ ]:
# Train the logistic regression model
lr_1.fit(X_train, y_train)

In [ ]:
# Check how many iterations the logistic regression model required to converge to a solution
lr_1.n_iter_

### Test Logistic Regression model 1

In [ ]:
# Generate predictions against the test set
y_pred = lr_1.predict(X_test)

### Evaluate Logistic Regression model 1

In [ ]:
# Calculate model accuracy
accuracy = lr_1.score(X_test, y_test)
print("accuracy =", round((accuracy * 100), 2), "%")

In [ ]:
# classification report
print("Classification Report:\n")

print(classification_report(
    y_test,
    y_pred,
    target_names=le.classes_
))

In [ ]:
#do we have a data leak?
result = permutation_importance(lr_1, X_test, y_test, n_repeats=5,
                                random_state=42, scoring="f1_macro", n_jobs=-1)
perm = pd.Series(result.importances_mean, index=X_test.columns).sort_values(ascending=False)
print(perm.head(15))

In [ ]:
# confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Display confusion matrix
plt.figure(figsize=(7, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=le.classes_,
    yticklabels=le.classes_
)

plt.title('Confusion Matrix - Logistic Regression')
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.tight_layout()
plt.show()

In [ ]:
# ROC curve
y_prob = lr_1.predict_proba(X_test)[:, 1]

# Calculate ROC curve values
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

# Calculate Area Under the Curve (AUC)
roc_auc = roc_auc_score(y_test, y_prob)

# Plot ROC curve
plt.figure(figsize=(8, 6))

plt.plot(
    fpr,
    tpr,
    label=f'Logistic Regression (AUC = {roc_auc:.3f})'
)

plt.plot([0, 1], [0, 1], linestyle='--', label='Random Classifier')

plt.title('ROC Curve - Logistic Regression')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

##Train Random Forest model 1

In [ ]:
rf_1 = RandomForestClassifier(random_state=42)

print('RandomForestClassifier default hyperparameters\n')
rf_1.get_params()

In [ ]:
%%time

rf_1.fit(X_train, y_train)

### Test Random Forest model 1

In [ ]:
# Predict using the test set
y_pred = rf_1.predict(X_test)

### Evaluate Random Forest model 1

In [ ]:
# Display baseline accuracy score
baseline_accuracy = round((rf_1.score(X_test, y_test) * 100), 3)
print(f"Baseline accuracy = {baseline_accuracy}%")

In [ ]:
# Display classification report
print(classification_report(y_test, y_pred))

In [ ]:
# Generate confusion matrix
cnf_matrix = confusion_matrix(y_test, y_pred)
cnf_matrix

In [ ]:
# Display confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cnf_matrix,
                              display_labels=rf_1.classes_)
sns.heatmap(
    cnf_matrix,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=le.classes_,
    yticklabels=le.classes_
)

plt.title('Confusion Matrix - Random Forest')
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.tight_layout()
plt.show()

##Train Gradient Boosting model 1

In [ ]:
# Instantiate GradientBoostingClassifier with default hyperparameters
gb_1 = GradientBoostingClassifier(random_state = 42)
gb_1.get_params()

In [ ]:
%%time

# Fit GradientBoostingClassifier
gb_1.fit(X_train_sm, y_train_sm)

### Test Gradient Boosting model 1

In [ ]:
# Evaluate the model with Default Params
#Generate predictions for the baseline model
y_pred = gb_1.predict(X_test)

### Evaluate Gradient Boosting model 1

In [ ]:
# Display accuracy score
accuracy = round((gb_1.score(X_test, y_test) * 100), 3)
print(f"Accuracy = {accuracy}%")

In [ ]:
# Display classification report
print(classification_report(y_test, y_pred))

In [ ]:
# Generate confusion matrix
cnf_gb_matrix = confusion_matrix(y_test, y_pred)

# Display confusion matrix with class labels
disp = ConfusionMatrixDisplay(confusion_matrix=cnf_gb_matrix,
                              display_labels=gb_1.classes_)
disp.plot(cmap=plt.cm.Blues, values_format='d')
plt.show()

In [ ]:
# Display ROC curve
#  Receiver Operating Characteristic(ROC) curve is a plot of the true positive rate
#   against the false positive rate
#  It shows the tradeoff between sensitivity and specificity
y_pred_proba = gb_1.predict_proba(X_test)[::,1]
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
auc = round(roc_auc_score(y_test, y_pred_proba),4)
gini = round(((auc * 2) - 1),4)
label_string = f'AUC={auc}, Gini={gini}'
plt.plot(fpr, tpr, label=label_string)
plt.legend(loc=4)
plt.title('ROC curve', y=1.05)
plt.ylabel('True positive rate')
plt.xlabel('False positive rate')
plt.show()

##Train Support Vector Classifier model 1


In [ ]:
SVC_1 = SVC(probability= True,random_state=42)
SVC_1.get_params()
#maybe mess with the probability??


In [ ]:
#train SVC
# was very worried because the wall time freaked out on me
%%time

SVC_1.fit(X_train, y_train)

### Test SVC model 1

In [128]:
# Generate predictions for the baseline SVC
y_pred_baseline = SVC_1.predict(X_test)

# Calculate and display the baseline accuracy
accuracy_baseline = accuracy_score(y_test, y_pred_baseline)
print(f"SVC Accuracy (Baseline): {accuracy_baseline:.4}")

KeyboardInterrupt: 

### Evaluate SVC model 1

In [ ]:
# Generate confusion matrix
svc_matrix = confusion_matrix(y_test, y_pred_baseline)

# Display confusion matrix with class labels
disp = ConfusionMatrixDisplay(confusion_matrix=svc_matrix,
                              display_labels=SVC_1.classes_)
disp.plot(cmap=plt.cm.Blues, values_format='d')
plt.show()

In [ ]:
y_pred_proba = SVC_1.predict_proba(X_test)[::,1]
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
auc = round(roc_auc_score(y_test, y_pred_proba),4)
gini = round(((auc * 2) - 1),4)
label_string = f'AUC={auc}, Gini={gini}'
plt.plot(fpr, tpr, label=label_string)
plt.legend(loc=4)
plt.title('ROC curve', y=1.05)
plt.ylabel('True positive rate')
plt.xlabel('False positive rate')
plt.show()

###Class Rebalancing using SMOTE

In [ ]:
# Display distribution counts for target variable 'Likely to Change Occupation'
print("Before SMOTE, counts of label 'Likely - 0': {}".format(sum(y==0)))
print("Before SMOTE, counts of label 'Unlikely - 1': {}\n".format(sum(y==1)))

print('Balance of positive and negative classes (%) before rebalancing:')
y_train.value_counts(normalize=True) * 100

In [ ]:
# Initialize SMOTE
sm = SMOTE(sampling_strategy=0.3, random_state=42) #  ratio of minority class set to 30% of dataset
sm.get_params()  # parameters (default of 5 close datapoints to work with, can change hyperparameter)

In [ ]:
# Fit and apply SMOTE only to training set
X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)

In [ ]:
print("After SMOTE, counts of label 'Likely - 0': {}".format(sum(y_train_sm==0)))
print("After SMOTE, counts of label 'Unlikely - 1': {}\n".format(sum(y_train_sm==1)))

print('Balance of positive and negative classes (%) after rebalancing:')
y_train_sm.value_counts(normalize=True) * 100

####Class Redistribution using SMOTE:

'Likely' Class 0:
- before: 91.77%
  - after: 76.92%

'Unlikely' Class 1:
- before: 8.23%
  - after: 23.08%

# Second Model Iteration with SMOTE

##Train Logistic Regression model 2 (SMOTE)

In [ ]:
# Fit on the resampled train set
lr_2 = LogisticRegression(random_state=42)

%%time
lr_2.fit(X_train_sm, y_train_sm)       # pass in rebalanced data

In [ ]:
# Check how many iterations the logistic regression model required to converge to a solution
lr_2.n_iter_

In [ ]:
# Generate predictions against the test set
y_pred = lr_2.predict(X_test)

### Evaluate Logistic Regression model 2

In [ ]:
# Calculate model accuracy
accuracy = lr_2.score(X_test, y_test)
print("accuracy =", round((accuracy * 100), 2), "%")

In [ ]:
# classification report
print("Classification Report:\n")

print(classification_report(
    y_test,
    y_pred,
    target_names=le.classes_
))

In [ ]:
# confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Display confusion matrix
plt.figure(figsize=(7, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=le.classes_,
    yticklabels=le.classes_
)

plt.title('Confusion Matrix - Logistic Regression')
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.tight_layout()
plt.show()

In [ ]:
# ROC curve
y_prob = lr_2.predict_proba(X_test)[:, 1]

# Calculate ROC curve values
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

# Calculate Area Under the Curve (AUC)
roc_auc = roc_auc_score(y_test, y_prob)

# Plot ROC curve
plt.figure(figsize=(8, 6))

plt.plot(
    fpr,
    tpr,
    label=f'Logistic Regression (AUC = {roc_auc:.3f})'
)

plt.plot([0, 1], [0, 1], linestyle='--', label='Random Classifier')

plt.title('ROC Curve - Logistic Regression')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

##Train Random Forest model 2 (SMOTE)

In [ ]:
# Fit on the resampled train set
rf_2 = RandomForestClassifier(random_state=42)

%%time
rf_2.fit(X_train_sm, y_train_sm)       # pass in rebalanced data

In [ ]:
# Predict using the test set
y_pred = rf_2.predict(X_test)

### Evaluate Random Forest model 2

In [ ]:
# Display baseline accuracy score
baseline_accuracy = round((rf_2.score(X_test, y_test) * 100), 3)
print(f"Baseline accuracy = {baseline_accuracy}%")

In [ ]:
# Display classification report
print(classification_report(y_test, y_pred))

In [ ]:
# Generate confusion matrix
cnf_matrix = confusion_matrix(y_test, y_pred)
cnf_matrix

In [ ]:
# Display confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cnf_matrix,
                              display_labels=rf_2.classes_)
sns.heatmap(
    cnf_matrix,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=le.classes_,
    yticklabels=le.classes_
)

plt.title('Confusion Matrix - Random Forest')
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.tight_layout()
plt.show()

## Train Gradient Boosting model 2 (SMOTE)

In [ ]:
# Fit on the resampled train set
gb_2 = GradientBoostingClassifier(random_state = 42)

%%time
gb_2.fit(X_train_sm, y_train_sm)       # pass in rebalanced data

In [ ]:
# Evaluate the model with Default Params
# Generate predictions for the baseline model
y_pred = gb_2.predict(X_test)

### Evaluate Gradient Boosting model 2

In [ ]:
# Display accuracy score
accuracy = round((gb_2.score(X_test, y_test) * 100), 3)
print(f"Accuracy = {accuracy}%")

In [ ]:
# Display classification report
print(classification_report(y_test, y_pred))

In [ ]:
# Generate confusion matrix
cnf_gb_matrix = confusion_matrix(y_test, y_pred)

# Display confusion matrix with class labels
disp = ConfusionMatrixDisplay(confusion_matrix=cnf_gb_matrix,
                              display_labels=gb_2.classes_)
disp.plot(cmap=plt.cm.Blues, values_format='d')
plt.show()

In [ ]:
# Display ROC curve
#  Receiver Operating Characteristic(ROC) curve is a plot of the true positive rate
#   against the false positive rate
#  It shows the tradeoff between sensitivity and specificity
y_pred_proba = gb_2.predict_proba(X_test)[::,1]
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
auc = round(roc_auc_score(y_test, y_pred_proba),4)
gini = round(((auc * 2) - 1),4)
label_string = f'AUC={auc}, Gini={gini}'
plt.plot(fpr, tpr, label=label_string)
plt.legend(loc=4)
plt.title('ROC curve', y=1.05)
plt.ylabel('True positive rate')
plt.xlabel('False positive rate')
plt.show()

## Train SCV model 2 (SMOTE)

In [ ]:
SVC_2 = SVC(probability= True,random_state=42)

%%time
SVC_2.fit(X_train_sm, y_train_sm)

In [ ]:
# Generate predictions for the baseline SVC
y_pred_baseline = SVC_2.predict(X_test)

# Calculate and display the baseline accuracy
accuracy_baseline = accuracy_score(y_test, y_pred_baseline)
print(f"SVC Accuracy (Baseline): {accuracy_baseline:.4}")

### Evaluate SVC model 2

In [ ]:
# Generate confusion matrix
svc_matrix = confusion_matrix(y_test, y_pred_baseline)

# Display confusion matrix with class labels
disp = ConfusionMatrixDisplay(confusion_matrix=svc_matrix,
                              display_labels=SVC_2.classes_)
disp.plot(cmap=plt.cm.Blues, values_format='d')
plt.show()

In [ ]:
y_pred_proba = SVC_2.predict_proba(X_test)[::,1]
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
auc = round(roc_auc_score(y_test, y_pred_proba),4)
gini = round(((auc * 2) - 1),4)
label_string = f'AUC={auc}, Gini={gini}'
plt.plot(fpr, tpr, label=label_string)
plt.legend(loc=4)
plt.title('ROC curve', y=1.05)
plt.ylabel('True positive rate')
plt.xlabel('False positive rate')
plt.show()

# Third Model Iteration with Hypertuning

## Hyperparameter Tuning: RandomForest


commented out for now before we decide which we wanna go with


#ideas:

*    class weight
*    max_depth (address overfitting)
*   max_features (tree correlation)
*   n_estimators? (stability?)





In [ ]:
# Create the hyperparameter grid

#grid = {'max_depth': max_depth,
        #'min_samples_leaf': min_samples_leaf,
        #'max_features': max_features,
        #'class_weight': [None, 'balanced']
       #}

In [ ]:
#Print the Hyperparam grid

#pprint(grid)

In [ ]:
# Set up grid search with k-fold cross validation using the hyperparameter grid

# Grid search of hyperparameters using 5 fold cross validation.
#gs = GridSearchCV(rf_1, param_grid=grid, cv=5,
                  #scoring="accuracy", return_train_score=True,
                  #n_jobs=-1, verbose=1)


In [ ]:
#Perform Grid Search

#%%time

#gs.fit(X_train_sm, y_train_sm)

In [ ]:
#Print the best Params found by grid search


#gs_best_params = gs.best_params_
#print("Optimal hyperparameter combination:", gs_best_params)
#print("\nMean cross-validated training accuracy score:", np.round(gs.best_score_,4))

In [ ]:
#personal exploration from Nats HW could be something we look at for feature importance
#coudl already be here???

#feature_names = X.columns
#importances = pd.Series(best_estimator.feature_importances_, index=feature_names)

#plt.figure(figsize=(8, 6))
#sns.barplot(x=importances.values, y=importances.index)
#plt.title('Random Forest Feature Importance')
#plt.xlabel('Importance')
#plt.show()

In [ ]:
#best estimator

#best_estimator = gs.best_estimator_
#best_estimator.get_params()

In [ ]:
# Make predictions on test set with best estimator

#y_pred = best_estimator.predict(X_test)

In [ ]:
# Display baseline accuracy score

#best_grid_search_accuracy = round((best_estimator.score(X_test, y_test) * 100), 3)
#print(f"Baseline model accuracy = {baseline_accuracy}%")
#print(f"Best model after grid search accuracy = {best_grid_search_accuracy}%")

In [ ]:
# Display classification report for best estimator

#print(classification_report(y_test, y_pred))

In [ ]:
# Generate confusion matrix for best estimator

#cm = confusion_matrix(y_test, y_pred)

In [ ]:
# Display confusion matrix with class labels for best estimator

#disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              #display_labels=best_estimator.classes_)
#disp.plot()
#plt.show()

In [ ]:
# Define the models and their corresponding accuracy values

#models = ['Baseline RF', 'Grid Search Best RF']
#accuracies = [baseline_accuracy, best_grid_search_accuracy]


#plt.figure(figsize=(7, 4))
#ax = sns.barplot(x=models, y=accuracies)
#ax.bar_label(ax.containers[0], fmt='%.2f%%')
#plt.title('Test Accuracy Comparison')
#plt.xlabel('Model')
#plt.ylabel('Accuracy (%)')
#plt.ylim(0, 100)
#plt.show()

##Hyperparameter Tuning: Gradient Boosting


# Ideas:


*   max_depth (capture more than 3 interactions)

*   max_features (decorrelaction)



In [ ]:
# Fit on the resampled train set
#need to add more hyperparams
gb_3 = GradientBoostingClassifier(random_state=42)
gb_3.fit(X_train_sm, y_train_sm)

In [ ]:
#%%time

# Retrain the model

#gb_3.fit(X_train_sm, y_train_sm)

In [ ]:
# Generate predictions for the retrained model

#y_pred = gb_3.predict(X_test)

In [ ]:
# Display accuracy score

#accuracy = round((gb_3.score(X_test, y_test) * 100), 3)
#print(f"Accuracy = {accuracy}%")

In [ ]:
# Display classification report

#print(classification_report(y_test, y_pred))

In [ ]:
# Gerenate confusion matrix

#cnf_matrix = confusion_matrix(y_test, y_pred)

# Display confusion matrix with class labels
#disp = ConfusionMatrixDisplay(confusion_matrix=cnf_matrix,
                              #display_labels=gb_3.classes_)
##disp.plot(cmap=plt.cm.Blues, values_format='d')
#plt.show()

In [ ]:
# Display ROC curve
#  Receiver Operating Characteristic(ROC) curve is a plot of the true positive rate
#   against the false positive rate
#  It shows the tradeoff between sensitivity and specificity
#y_pred_proba = gb_3.predict_proba(X_test)[::,1]
#fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
#auc = round(roc_auc_score(y_test, y_pred_proba),4)
#gini = round(((auc * 2) - 1),4)
#label_string = f'AUC={auc}, Gini={gini}'
#plt.plot(fpr, tpr, label=label_string)
#plt.legend(loc=4)
#plt.title('ROC curve', y=1.05)
#plt.ylabel('True positive rate')
#plt.xlabel('False positive rate')
#plt.show()

##HyperParameter Tuning: SVC


# Ideas:


*   why is it taking so long? explore probability False

*   List item
*   List item


*   max_features (decorrelaction)

In [ ]:
%%time

# Fit on the resampled train set
#need to add more hyperparams
SVC_3 = SVC(probability= True, random_state=42)
SVC_3.fit(X_train_sm, y_train_sm)


In [ ]:
# Generate predictions for re-trained model

#y_pred = SVC_3.predict(X_test)


# Calculate and display accuracy for re-trained model


#accuracy_baseline = accuracy_score(y_test, y_pred)
#print(f"SVC Accuracy (re-trained model): {accuracy_baseline:.4}")

In [ ]:
# Generate confusion matrix
#svc_matrix = confusion_matrix(y_test, y_pred)

# Display confusion matrix with class labels
#disp = ConfusionMatrixDisplay(confusion_matrix=svc_matrix,
                              #display_labels=SVC_2.classes_)
#disp.plot(cmap=plt.cm.Blues, values_format='d')
#plt.show()

In [ ]:
# Display ROC curve